## ML_1038: Mixture of Experts (MoE)

**Difficulty**: Hard | **TorchCode**: #28

### Problem
Implement a Mixtral-style Mixture of Experts layer. A router selects the top-k experts per token, computes their outputs, and takes a softmax-weighted sum.

### Formula
$$y = \sum_{i \in \text{top-k}} \text{softmax}(r_i) \cdot E_i(x), \quad r = W_{\text{router}}\, x$$

In [ ]:
import torch
import torch.nn as nn

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(num_experts)
        ])

    def forward(self, x):
        orig_shape = x.shape
        x_flat = x.reshape(-1, x.shape[-1]) if x.dim() == 3 else x
        logits = self.router(x_flat)
        top_vals, top_idx = logits.topk(self.top_k, dim=-1)
        weights = torch.softmax(top_vals, dim=-1)
        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for e in range(len(self.experts)):
                mask = (top_idx[:, k] == e)
                if mask.any():
                    output[mask] += weights[mask, k:k+1] * self.experts[e](x_flat[mask])
        return output.reshape(orig_shape)

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Output shape ---
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
assert isinstance(moe, nn.Module)
assert moe(torch.randn(2, 8, 32)).shape == (2, 8, 32)

# --- Test 2: Has router and experts ---
assert hasattr(moe, 'router') and hasattr(moe, 'experts')
assert len(moe.experts) == 4

# --- Test 3: Router logits shape ---
moe2 = MixtureOfExperts(16, 32, num_experts=8, top_k=2)
assert moe2.router(torch.randn(4, 16)).shape == (4, 8)

# --- Test 4: Gradient flow ---
moe3 = MixtureOfExperts(16, 32, num_experts=4, top_k=2)
x = torch.randn(1, 4, 16, requires_grad=True)
moe3(x).sum().backward()
assert x.grad is not None

print('All tests passed!')